# SoS Phase 2 — Document Ingestion Pipeline

This notebook ingests the 501 crawled regulatory PDFs into OpenSearch.

**Pipeline:** PDF → Extract Text → Chunk → Compress (Bedrock LLM) → Embed (Titan) → Index (OpenSearch)

**Prerequisites:**
- SageMaker execution role with Bedrock + OpenSearch access
- Documents uploaded to S3 (`s3://ms-sos-legal-documents/crawled-documents/`)
- OpenSearch domain running and accessible

## 1. Install Dependencies

In [ ]:
!pip install -q PyMuPDF opensearch-py requests-aws4auth pydantic python-dotenv boto3

## 2. Configure Environment

In [ ]:
import os
import sys

# Add Sagemaker Files to path so imports work
sys.path.insert(0, '.')

# Configuration
os.environ['USE_AWS'] = 'true'
os.environ['AWS_REGION'] = 'us-east-1'
os.environ['OPENSEARCH_ENDPOINT'] = 'https://search-ms-sos-legal-vectors-f34ac65fpnys3mjc7xuzvawcdu.us-east-1.es.amazonaws.com'
os.environ['OPENSEARCH_INDEX'] = 'ms-legal-abstracts'
os.environ['BEDROCK_LLM_MODEL'] = 'us.anthropic.claude-sonnet-4-6'  # Claude Sonnet 4.6 via inference profile
os.environ['BEDROCK_EMBEDDING_MODEL'] = 'amazon.titan-embed-text-v2:0'

# S3 settings
S3_BUCKET = 'ms-sos-legal-documents'
S3_PREFIX = 'crawled-documents'

print('Environment configured')
print(f'  Region: {os.environ["AWS_REGION"]}')
print(f'  OpenSearch: {os.environ["OPENSEARCH_ENDPOINT"]}')
print(f'  LLM: {os.environ["BEDROCK_LLM_MODEL"]}')
print(f'  Embedding: {os.environ["BEDROCK_EMBEDDING_MODEL"]}')
print(f'  S3: s3://{S3_BUCKET}/{S3_PREFIX}/')

## 3. Upload Crawled Documents to S3

Run this once to upload the local `crawled_documents/` folder to S3.
Skip if documents are already in S3.

In [ ]:
# Upload local crawled_documents/ to S3 (run once)
# Uncomment and run if you need to upload from this notebook

# import boto3
# from pathlib import Path
# 
# s3 = boto3.client('s3')
# local_root = Path('../crawled_documents')  # adjust path
# 
# uploaded = 0
# for pdf in sorted(local_root.rglob('*.pdf')):
#     rel = pdf.relative_to(local_root)
#     s3_key = f'{S3_PREFIX}/{rel}'
#     s3.upload_file(str(pdf), S3_BUCKET, s3_key)
#     uploaded += 1
#     if uploaded % 50 == 0:
#         print(f'  Uploaded {uploaded} files...')
# 
# print(f'Done: {uploaded} PDFs uploaded to s3://{S3_BUCKET}/{S3_PREFIX}/')

## 4. Verify S3 Contents

In [ ]:
from ingest_pipeline import list_s3_pdfs
import boto3

s3 = boto3.client('s3')
pdfs = list_s3_pdfs(s3, S3_BUCKET, S3_PREFIX)

print(f'Total PDFs in S3: {len(pdfs)}')
print()

# Count by state and agency
from collections import Counter
by_state = Counter(p['state'] for p in pdfs)
by_agency = Counter(p['agency_type'] for p in pdfs)

print('By state:')
for state, count in sorted(by_state.items()):
    print(f'  {state}: {count}')

print('\nBy agency:')
for agency, count in sorted(by_agency.items()):
    print(f'  {agency}: {count}')

## 5. Run the Ingestion Pipeline

Process all documents or filter by state/agency.

**Estimated time:** ~2-3 minutes per document (Bedrock compression is the bottleneck).
For 501 documents, expect ~15-20 hours for a full run. Consider running per-state.

In [ ]:
from ingest_pipeline import IngestionPipeline, PipelineConfig

pipeline_config = PipelineConfig(
    s3_bucket=S3_BUCKET,
    s3_prefix=S3_PREFIX,
    chunk_size=2000,
    chunk_overlap=200,
    batch_size=20,
)

pipeline = IngestionPipeline(pipeline_config=pipeline_config)

In [ ]:
# Start with Mississippi (smallest — 17 docs) to verify everything works
stats_ms = pipeline.run(states=['MS'])

In [ ]:
# Then run remaining states one at a time (restart pipeline between states to reset stats)
for state in ['TN', 'AL', 'LA', 'AR', 'GA', 'TX']:
    print(f'\n{"="*60}')
    print(f'Starting {state}...')
    print(f'{"="*60}')
    state_stats = pipeline.run(states=[state])
    print(f'\n{state} done: {len(state_stats.documents_processed)} docs, '
          f'{state_stats.total_abstracts} abstracts, '
          f'{len(state_stats.failed_documents)} failures')

## 6. Verify OpenSearch Index

In [ ]:
from vector_store_opensearch import OpenSearchVectorStore

vs = OpenSearchVectorStore()
stats = vs.get_stats()

print(f'Index: {stats["index_name"]}')
print(f'Total abstracts: {stats["total_abstracts"]}')
print(f'Unique documents: {stats["unique_documents"]}')
print(f'\nSample documents:')
for doc in stats['documents'][:10]:
    print(f'  {doc}')

## 7. Test Search

In [ ]:
# Test hybrid search
results = vs.search('dental licensing fees', top_k=5)

for r in results:
    a = r.abstract
    state = getattr(a, 'state', '??')
    print(f'[{state}] {a.source_document} (score: {r.similarity_score:.4f})')
    print(f'  {a.abstract_text[:150]}...')
    if hasattr(a, 'fee_amounts') and a.fee_amounts:
        for fee in a.fee_amounts:
            print(f'  Fee: ${fee.amount} ({fee.fee_type}) — {fee.description}')
    print()

In [ ]:
# Test cross-state search
results = vs.search(
    'reciprocity out-of-state licensure',
    top_k=10,
    filter_agency_type='medical',
)

for r in results:
    a = r.abstract
    state = getattr(a, 'state', '??')
    print(f'[{state}] {a.source_document} — {a.section_identifier or ""}')
    recip = getattr(a, 'reciprocity_provisions', None)
    if recip:
        print(f'  Reciprocity: {recip[:200]}')
    print()

In [ ]:
# Test fee aggregation
fee_aggs = vs.aggregate_fees(filter_agency_type='dental')

for state_bucket in fee_aggs.get('by_state', {}).get('buckets', []):
    state = state_bucket['key']
    print(f'\n{state}:')
    for fee_bucket in state_bucket['fees']['by_type']['buckets']:
        fee_type = fee_bucket['key']
        avg = fee_bucket['avg_amount']['value']
        print(f'  {fee_type}: avg=${avg:.2f}')